In [1]:
import geopy.distance
import pandas as pd
import numpy as np
from glob import glob
from os.path import join
from multiprocessing import Pool
import pyarrow.parquet as pq
from skmob.core.trajectorydataframe import *
from skmob.measures import individual
from skmob.preprocessing import filtering, compression, detection, clustering
from skmob.utils import utils, constants
from src.clustering_st import cluster

In [2]:
base_data_path = ""
key_file = ""
daily_files = sorted(glob(join(base_data_path, "veraset*")))

In [3]:
## load the metadata
meta = [pq.ParquetFile(x).metadata for x in daily_files]

In [4]:
def get_row_groups(row_group, files):
    """ Read a specific row group across all files. """
    
    return pd.concat([pq.ParquetFile(f, metadata=m).read_row_group(row_group).to_pandas() for f, m in zip(files, meta)]) 


def get_row_group_trajectories(row_group, files):
    """ Read the row group from all files and return a skmobility Trajectory DataFrame """
    df = get_row_groups(row_group, files)
    tdf = TrajDataFrame(df.reset_index(), latitude='latitude', longitude='longitude', datetime='datetime', user_id='caid')
    
    return tdf

In [5]:
row_groups = np.arange(10, 20) # grab the first thousand row groups (1 million users)
files = [daily_files] * len(row_groups)
args = [(rg, f) for rg, f in zip(row_groups, files)]

In [6]:
with Pool(processes=8) as pool:
    data = pd.concat(pool.starmap(get_row_group_trajectories, args))

In [7]:
data

,uid,datetime,id_type,lat,lng,horizontal_accuracy
0,000b8615d230577417bfb3c1115e13088c5e9199fead13...,2024-03-14 17:50:11-07:00,idfa,34.179401,-118.296379,12
1,000b8615d230577417bfb3c1115e13088c5e9199fead13...,2024-03-14 17:50:12-07:00,idfa,34.179401,-118.296341,12
2,000b8615d230577417bfb3c1115e13088c5e9199fead13...,2024-03-14 17:50:14-07:00,idfa,34.179401,-118.296371,12
3,000b8615d230577417bfb3c1115e13088c5e9199fead13...,2024-03-14 18:02:12-07:00,idfa,34.179379,-118.296379,9
4,000b8615d230577417bfb3c1115e13088c5e9199fead13...,2024-03-14 18:02:13-07:00,idfa,34.179379,-118.296303,9
...,...,...,...,...,...,...
86870,00163785f66b80ff9850f6f99035728028c807a5d10a19...,2024-04-30 08:38:44-07:00,idfa,40.548241,-122.336723,11
86871,00163785f66b80ff9850f6f99035728028c807a5d10a19...,2024-04-30 08:38:47-07:00,idfa,40.548199,-122.336700,11
86872,00163785f66b80ff9850f6f99035728028c807a5d10a19...,2024-04-30 08:38:48-07:00,idfa,40.548241,-122.336739,11
86873,00163785f66b80ff9850f6f99035728028c807a5d10a19...,2024-04-30 08:39:02-07:00,idfa,40.548199,-122.336700,11


In [8]:
data_uid = data[data[constants.UID] == data[constants.UID].mode()[0]].drop("uid", axis=1)
data_uid

,datetime,id_type,lat,lng,horizontal_accuracy
34187,2024-04-07 18:20:29-07:00,idfa,37.610802,-122.392998,1414
34188,2024-04-07 18:31:29-07:00,idfa,37.613602,-122.388702,57
34189,2024-04-07 18:35:29-07:00,idfa,37.613701,-122.388496,35
34190,2024-04-07 19:14:51-07:00,idfa,37.547001,-122.309402,4
34191,2024-04-07 21:03:35-07:00,idfa,37.597198,-120.925003,4
...,...,...,...,...,...
116555,2024-04-30 15:57:24-07:00,idfa,37.597198,-120.925400,5
116556,2024-04-30 15:57:32-07:00,idfa,37.597198,-120.925301,5
116557,2024-04-30 15:57:40-07:00,idfa,37.597198,-120.925201,5
116558,2024-04-30 15:57:47-07:00,idfa,37.597198,-120.925102,5


ZAO Algorithm
 - 1 entry per day during evacuation study period
 - for points with spatial inaccuracy > 800m, points with a velocity 80mph or higher have been discarded
 - remove the oscillation sequences  (jumping quickly back and forth between two coords) via time-window-based method [https://www.sciencedirect.com/science/article/pii/S0968090X17303637]
 - Remove location records that indicate travel between locations to obtain static location records.
 - Apply threshold-based recursive method to static location records to create check-in data.

> To begin with, the first location record L1 is taken as the check-in group G1, then the distance D12 between L1 to the next location record L2 is calculated. If D12 is smaller than the threshold (300 meters), then L2 is added to the check-in group G1. Otherwise, L2 is taken as a new check-in group G2. The same process is repeated for each static location record, so that every static location record is assigned to a check-in group. Next, for each check-in group, the centroid is defined as the mean value of all latitudes and longitudes in this group, and the centroid itself is taken as a check-in point. The duration of a check-in is defined as the difference between the timestamp of the last location record and the timestamp of the first location record in the check-in group. Finally, check-ins with duration smaller than the 10-minute threshold are removed, because these check-ins indicate a temporary stop such as waiting for the traffic lights or picking someone up.  

- DBSCAN applied to check-in data.
- The largest and second largest DBSCAN clusters are assumed to be the home and work locations respectively.

In [9]:
def filter_by_min_pings_per_daterange(tdf, start_date='2024-03-14', end_date='2024-04-30', min_pings=1):

    # filter by date range, inclusive    
    _tdf = tdf.copy(deep=True)
    _tdf.loc[:, constants.DATETIME] = _tdf[constants.DATETIME].dt.normalize()
    _tdf = _tdf[(_tdf[constants.DATETIME] >= start_date) & (_tdf[constants.DATETIME] <= end_date)]
    threshold = (len(_tdf[constants.DATETIME].unique()) - 1) * min_pings
    
    # if 'uid' column in not present in the TrajDataFrame
    if constants.UID not in _tdf.columns:
        _tdf = _tdf.set_index(constants.DATETIME, drop=True)
        _tdf = _tdf.loc[:, constants.LATITUDE].groupby(level=0).count()
        if len(_tdf[_tdf > 1]) > threshold:
            return tdf
    else:
        _tdf = _tdf.set_index([constants.UID, constants.DATETIME], drop=True)
        _tdf = _tdf.loc[:, constants.LATITUDE].groupby(level=[0,1]).count()
        _tdf[_tdf > 1] = 1
        _tdf = _tdf.groupby(level=0).count()
        uids = _tdf[_tdf > threshold].index
        
        return tdf[tdf[constants.UID].isin(uids)]

In [10]:
traj = filter_by_min_pings_per_daterange(data, start_date='2024-03-28', end_date='2024-03-30', min_pings=1)
traj

,uid,datetime,id_type,lat,lng,horizontal_accuracy
91,000bd7b3cc6ac11b540a3cee26de64da3b3d9a20d9f048...,2024-03-14 19:55:50-07:00,idfa,34.439678,-118.514778,9
92,000bd7b3cc6ac11b540a3cee26de64da3b3d9a20d9f048...,2024-03-14 19:55:53-07:00,idfa,34.439659,-118.514740,9
93,000bd7b3cc6ac11b540a3cee26de64da3b3d9a20d9f048...,2024-03-14 19:55:56-07:00,idfa,34.439671,-118.514717,9
94,000bd7b3cc6ac11b540a3cee26de64da3b3d9a20d9f048...,2024-03-14 19:56:55-07:00,idfa,34.445400,-118.513763,8
95,000bd7b3cc6ac11b540a3cee26de64da3b3d9a20d9f048...,2024-03-14 19:56:56-07:00,idfa,34.445431,-118.513771,8
...,...,...,...,...,...,...
86865,00166479b68d2a4abe80466d865ca6e735811ab37b5c8a...,2024-04-30 12:21:08-07:00,aaid,33.955502,-118.286301,11
86866,00166479b68d2a4abe80466d865ca6e735811ab37b5c8a...,2024-04-30 16:38:34-07:00,aaid,34.047199,-117.473099,0
86867,00166479b68d2a4abe80466d865ca6e735811ab37b5c8a...,2024-04-30 16:56:10-07:00,aaid,34.047241,-117.473137,11
86868,00166479b68d2a4abe80466d865ca6e735811ab37b5c8a...,2024-04-30 16:56:13-07:00,aaid,34.047218,-117.473106,11


In [11]:
print(data.uid.unique().shape)
print(traj.uid.unique().shape)

(10000,)
(264,)


In [12]:
def filter_by_horizontal_accuracy_and_speed(tdf, max_horizontal_accuracy=800, max_speed_kmh=128.7):

    columns = [constants.LATITUDE, constants.LONGITUDE]
    distfunc = geopy.distance.distance
    
    tdf.reset_index(inplace=True)
    for i in tdf[tdf.horizontal_accuracy >= max_horizontal_accuracy].index:
        if i-1 in tdf.index and tdf.loc[i-1, "uid"] == tdf.loc[i, "uid"]:
            start = i - 1
            end = i
        elif i+1 in tdf.index and tdf.loc[i+1, "uid"] == tdf.loc[i, "uid"]:
            start = i
            end = i + 1
        else:
            # print(f"skipped and condition: {i-1 in tdf.index} and {tdf.loc[i-1, 'uid'] == tdf.loc[i, 'uid']} and {tdf.loc[i, 'horizontal_accuracy']}")
            continue
        
        dt = utils.diff_seconds(tdf.loc[start, constants.DATETIME], tdf.loc[end, constants.DATETIME])
        distance = distfunc(tdf.loc[start, columns].tolist(), tdf.loc[end, columns].tolist()).km
        
        if distance / dt * 3600. > max_speed_kmh:
            tdf = tdf.drop(i)
            # print("REJECTED", i, speed)
    
    return tdf.set_index("index", drop=True)

In [13]:
print(traj.shape)
traj = filter_by_horizontal_accuracy_and_speed(traj, max_horizontal_accuracy=800, max_speed_kmh=128.7)
print(traj.shape)
traj

(206769, 6)
(206747, 6)


,uid,datetime,id_type,lat,lng,horizontal_accuracy
index,,,,,,
91,000bd7b3cc6ac11b540a3cee26de64da3b3d9a20d9f048...,2024-03-14 19:55:50-07:00,idfa,34.439678,-118.514778,9
92,000bd7b3cc6ac11b540a3cee26de64da3b3d9a20d9f048...,2024-03-14 19:55:53-07:00,idfa,34.439659,-118.514740,9
93,000bd7b3cc6ac11b540a3cee26de64da3b3d9a20d9f048...,2024-03-14 19:55:56-07:00,idfa,34.439671,-118.514717,9
94,000bd7b3cc6ac11b540a3cee26de64da3b3d9a20d9f048...,2024-03-14 19:56:55-07:00,idfa,34.445400,-118.513763,8
95,000bd7b3cc6ac11b540a3cee26de64da3b3d9a20d9f048...,2024-03-14 19:56:56-07:00,idfa,34.445431,-118.513771,8
...,...,...,...,...,...,...
86865,00166479b68d2a4abe80466d865ca6e735811ab37b5c8a...,2024-04-30 12:21:08-07:00,aaid,33.955502,-118.286301,11
86866,00166479b68d2a4abe80466d865ca6e735811ab37b5c8a...,2024-04-30 16:38:34-07:00,aaid,34.047199,-117.473099,0
86867,00166479b68d2a4abe80466d865ca6e735811ab37b5c8a...,2024-04-30 16:56:10-07:00,aaid,34.047241,-117.473137,11


In [14]:
traj_noloops = filtering.filter(traj, max_speed_kmh=10000.0)
print(traj_noloops.shape)
traj_noloops = filtering.filter(traj_noloops, max_speed_kmh=10000.0, include_loops=True, speed_kmh=5.0, max_loop=6, ratio_max=0.25)
print(traj_noloops.shape)
traj_noloops

(205846, 6)
(195257, 6)


,uid,datetime,id_type,lat,lng,horizontal_accuracy
0,000b29b9f7e097d11c9e648503118c19143b85f04e68b7...,2024-03-15 00:05:18-07:00,idfa,33.948730,-116.460716,6
1,000b29b9f7e097d11c9e648503118c19143b85f04e68b7...,2024-03-15 19:17:02-07:00,idfa,33.946770,-116.501297,11
2,000b29b9f7e097d11c9e648503118c19143b85f04e68b7...,2024-03-15 19:17:08-07:00,idfa,33.946751,-116.501297,11
3,000b29b9f7e097d11c9e648503118c19143b85f04e68b7...,2024-03-15 23:47:24-07:00,idfa,33.948799,-116.460800,4
4,000b29b9f7e097d11c9e648503118c19143b85f04e68b7...,2024-03-15 23:47:30-07:00,idfa,33.948799,-116.460823,4
...,...,...,...,...,...,...
195252,00166479b68d2a4abe80466d865ca6e735811ab37b5c8a...,2024-04-30 12:21:08-07:00,aaid,33.955502,-118.286301,11
195253,00166479b68d2a4abe80466d865ca6e735811ab37b5c8a...,2024-04-30 16:38:34-07:00,aaid,34.047199,-117.473099,0
195254,00166479b68d2a4abe80466d865ca6e735811ab37b5c8a...,2024-04-30 16:56:10-07:00,aaid,34.047241,-117.473137,11
195255,00166479b68d2a4abe80466d865ca6e735811ab37b5c8a...,2024-04-30 16:56:13-07:00,aaid,34.047218,-117.473106,11


In [15]:
traj_noloops_maxspeed = filtering.filter(traj_noloops, max_speed_kmh=1000.0)
print(traj_noloops_maxspeed.shape)
print(traj_noloops_maxspeed.uid.unique().shape)
traj_noloops_maxspeed

(194132, 6)
(264,)


,uid,datetime,id_type,lat,lng,horizontal_accuracy
0,000b29b9f7e097d11c9e648503118c19143b85f04e68b7...,2024-03-15 00:05:18-07:00,idfa,33.948730,-116.460716,6
1,000b29b9f7e097d11c9e648503118c19143b85f04e68b7...,2024-03-15 19:17:02-07:00,idfa,33.946770,-116.501297,11
2,000b29b9f7e097d11c9e648503118c19143b85f04e68b7...,2024-03-15 19:17:08-07:00,idfa,33.946751,-116.501297,11
3,000b29b9f7e097d11c9e648503118c19143b85f04e68b7...,2024-03-15 23:47:24-07:00,idfa,33.948799,-116.460800,4
4,000b29b9f7e097d11c9e648503118c19143b85f04e68b7...,2024-03-15 23:47:30-07:00,idfa,33.948799,-116.460823,4
...,...,...,...,...,...,...
194127,00166479b68d2a4abe80466d865ca6e735811ab37b5c8a...,2024-04-30 12:21:08-07:00,aaid,33.955502,-118.286301,11
194128,00166479b68d2a4abe80466d865ca6e735811ab37b5c8a...,2024-04-30 16:38:34-07:00,aaid,34.047199,-117.473099,0
194129,00166479b68d2a4abe80466d865ca6e735811ab37b5c8a...,2024-04-30 16:56:10-07:00,aaid,34.047241,-117.473137,11
194130,00166479b68d2a4abe80466d865ca6e735811ab37b5c8a...,2024-04-30 16:56:13-07:00,aaid,34.047218,-117.473106,11


In [16]:
def home_location_zao(tdf, **kwargs):

    _tdf = detection.stay_locations(tdf,
                                    stop_radius_factor=0.5,
                                    minutes_for_a_stop=10.0,
                                    spatial_radius_km=0.2,
                                    leaving_time=True,
                                    no_data_for_minutes=1000000000000.0,
                                    min_speed_kmh=None)
    print(_tdf.head())
    _tdf = clustering.cluster(_tdf, cluster_radius_km=0.1, min_samples=1)
    _tdf["duration"] = _tdf["leaving_datetime"] - _tdf["datetime"]
    
    _tdf1 = _tdf.groupby(["uid", "cluster"])["duration"].sum()
    _tdf2 = _tdf.groupby(["uid", "cluster"])[["lat", "lng"]].mean()
    _tdfhw = pd.concat((_tdf1.to_frame(), _tdf2), axis=1, join="inner")
    _tdfhw = _tdfhw.sort_values(by = ['uid', 'duration'], ascending = [False, False])
    _tdfhw = _tdfhw.groupby("uid").head(2)

    home = _tdfhw.groupby("uid").head(1).droplevel(1)
    home = home.rename(columns={"duration": "duration_home"})
    work = _tdfhw.groupby("uid").tail(1).droplevel(1)
    work = work.rename(columns={"lat": "lat_work", "lng": "lng_work", "duration": "duration_work"})
    
    return home.merge(work, left_index=True, right_index=True)

In [17]:
home_locs = home_location_zao(traj_noloops_maxspeed)
home_locs

                                                 uid  \
0  000b29b9f7e097d11c9e648503118c19143b85f04e68b7...   
1  000b29b9f7e097d11c9e648503118c19143b85f04e68b7...   
2  000b29b9f7e097d11c9e648503118c19143b85f04e68b7...   
3  000b29b9f7e097d11c9e648503118c19143b85f04e68b7...   
4  000b29b9f7e097d11c9e648503118c19143b85f04e68b7...   

                   datetime id_type        lat         lng  \
0 2024-03-15 00:05:18-07:00    idfa  33.948730 -116.460716   
1 2024-03-15 19:17:02-07:00    idfa  33.946760 -116.501297   
2 2024-03-15 23:47:24-07:00    idfa  33.948799 -116.460747   
3 2024-03-19 22:07:40-07:00    aaid  33.946955 -116.463371   
4 2024-03-20 18:35:52-07:00    idfa  33.944510 -116.475197   

   horizontal_accuracy          leaving_datetime  
0                    6 2024-03-15 19:17:02-07:00  
1                   11 2024-03-15 23:47:24-07:00  
2                   19 2024-03-19 22:07:40-07:00  
3                   14 2024-03-20 18:34:01-07:00  
4                   11 2024-03-23 2

/glade/derecho/scratch/ggantos/tmp/ipykernel_128521/3565899281.py:11: DeprecationWarning: This method will be deprecated in version 2.0. Instead, use the dbscan() method.
  _tdf = clustering.cluster(_tdf, cluster_radius_km=0.1, min_samples=1)


,duration_home,lat,lng,duration_work,lat_work,lng_work
uid,,,,,,
00166479b68d2a4abe80466d865ca6e735811ab37b5c8af209b9b315e326203a,12 days 01:13:30,33.749911,-117.907110,7 days 10:50:21,34.077206,-117.450607
001653ceb12155c80c6a491ac653ec965cd110c10e807bab4b4305fe9b2f75f8,18 days 02:06:09,32.862440,-116.920944,10 days 23:52:08,32.831793,-117.149411
0016469216df68f46a6ed829c4f7687de42f457ac766d993c563f78fd909f499,22 days 06:02:20,34.171007,-118.830902,8 days 01:13:06,34.146230,-118.604335
00163cef207054cc0304a9e66ee18ad1cd5e793ce48285abe3fc13d08dfe143d,22 days 16:44:57,38.677013,-121.171219,5 days 08:36:53,39.214928,-121.641678
001636469d238c7f49440b325c23becbd41a1cf7ecb7e69b19c8e12020ad7d18,13 days 18:02:13,33.675647,-117.961510,11 days 22:06:34,33.675665,-117.960049
...,...,...,...,...,...,...
000b4789c6bfb86c76c61a2962f69ce8e862c547f25d47db7e54e861f7a7663d,33 days 07:50:45,37.455583,-121.876449,3 days 07:54:22,39.006767,-120.991257
000b4479eea1ab8bcac89cdd2a9d277c5877b37edd220a8ee1639d9541154936,31 days 03:40:59,32.967946,-117.186561,7 days 04:20:11,32.915943,-117.231232
000b3cbd64eeecee2796d0cd961ced23d2eda6aeaabe5fc94fb06ce6fc370f0f,33 days 20:06:07,37.691652,-121.921479,9 days 20:04:54,37.687364,-121.913152
